In [31]:
# compare the two models: GLM and LMM
# preform a LRT to compare the advantage of using LMM instead of GLM

In [32]:
import numpy as np
import pandas as pd

from scipy.stats import shapiro             # for normality testS
import statsmodels.api as sm                # for linear mixed-effects models (LMM)
import statsmodels.formula.api as smf       # for linear mixed-effects models (LMM)
from statsmodels.formula.api import mixedlm # for LMM
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt

In [33]:
# Load shape measures, SELECTED
curRoot = 'C'  # 'C' or 'D'
# NOTE sca7, 3 and 2 relabeled use CS instead of CSSyl, only sca1 use CSSyl as before
curRegion = 'CSSyl'  # CSSyl, CS or CSpreCS 
curSCA = 1      # !!! modify !!!

In [34]:
#####################  load the time 1+2 dataset  #####################

# combining time1 and time2, after relabelling and REDO for sca1
#curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\RELABEL_REDO\RELABEL_REDO_time1_2_ctl_sca{curSCA}\{curRegion}\combined_time1_2_min_ctl_sca{curSCA}.csv'
curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\RELABEL_REDO\RELABEL_REDO_time1_2_ctl_sca{curSCA}\{curRegion}\combined_time1_2_max_ctl_sca{curSCA}.csv'
#curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\RELABEL_REDO\RELABEL_REDO_time1_2_ctl_sca{curSCA}\{curRegion}\combined_time1_2_max_ctl_sca{curSCA}_BIOSCA.csv'
#curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\RELABEL_REDO\RELABEL_REDO_time1_2_ctl_sca{curSCA}\{curRegion}\combined_time1_2_min_ctl_sca{curSCA}_BIOSCA.csv'

combined = pd.read_csv(curPath)
combined.index = combined['subjName']

# for specific analysis
combined_only_SCA = combined[combined['SCA'] == curSCA]
combined_CAG = combined.dropna(subset=['CAG', 'Age_onset'])
combined_CAG_only_SCA = combined_CAG[combined_CAG['SCA'] == curSCA]

In [35]:
#####################  Note that the dataset must be the same for LRT  !!!!!!!!   ######################
#####################  load the time 1 only dataset  #####################

# separating time1 and time2
#curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\Combined_Select_CSV\{curRegion}_ctl_sca1_time1_min.csv'
curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\Combined_Select_CSV\{curRegion}_ctl_sca1_time1_max.csv'
#curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\Combined_Select_CSV\{curRegion}_ctl_sca1_time2_min.csv'
#curPath = rf'{curRoot}:\B_projWIP\proj_ataxia\Combined_Select_CSV\{curRegion}_ctl_sca1_time2_max.csv'

combined_oneTime = pd.read_csv(curPath)
combined_oneTime.index = combined_oneTime['subjName']
# for specific analysis
combined_only_SCA_oneTime = combined_oneTime[combined_oneTime['SCA'] == curSCA]
combined_CAG_oneTime = combined_oneTime.dropna(subset=['CAG', 'Age_onset'])
combined_CAG_only_SCA_oneTime = combined_CAG_oneTime[combined_CAG_oneTime['SCA'] == curSCA]

In [36]:
#############################  definging functions ################################

# For comparison of the same subject across time points, add a column 'ori_subjName' without the time point postfix
# Remove the postfix in the form of '_something'
def addOriSubj(df):
    df.loc[:,'ori_subjName'] = df['subjName'].str.replace(r'_.+$', '', regex=True)
    return df

# Remove subjects with only one time point
def removeSingleTimePointReIndex(df):
  valid_subjects = df['ori_subjName'].value_counts()   # ensure each subject has two time points
  df = df[df['ori_subjName'].isin(valid_subjects[valid_subjects > 1].index)]
  df = df.sort_values(by=["ori_subjName", "Time_point"]).reset_index(drop=True)    
  return df

# Centering values
def centering_values(df):
    df.loc[:,'CAG_centered'] = df['CAG'] - df['CAG'].mean()
    df.loc[:,'SARA_centered'] = df['SARA'] - df['SARA'].mean()
    df.loc[:,'CCFS_centered'] = df['CCFS'] - df['CCFS'].mean()
    df.loc[:,'INAS_centered'] = df['INAS'] - df['INAS'].mean()    
    df.loc[:,'Age_onset_centered'] = df['Age_onset'] - df['Age_onset'].mean()
    df.loc[:,'Age_centered'] = df['Age'] - df['Age'].mean()
    df.loc[:,'iso1_centered'] = df['iso1'] - df['iso1'].mean()   
    df.loc[:,'iso2_centered'] = df['iso2'] - df['iso2'].mean()       
    df.loc[:,'iso3_centered'] = df['iso3'] - df['iso3'].mean()   
    df.loc[:,'UMAP1_U1_centered'] = df['UMAP1_U1'] - df['UMAP1_U1'].mean()   
    df.loc[:,'UMAP1_U2_centered'] = df['UMAP1_U2'] - df['UMAP1_U2'].mean() 
    df.loc[:,'UMAP1_U3_centered'] = df['UMAP1_U3'] - df['UMAP1_U3'].mean() 
    df.loc[:,'UMAP2_U3_centered'] = df['UMAP2_U3'] - df['UMAP2_U3'].mean() 
    df.loc[:,'UMAP1_U4_centered'] = df['UMAP1_U4'] - df['UMAP1_U4'].mean() 
    df.loc[:,'UMAP2_U4_centered'] = df['UMAP2_U4'] - df['UMAP2_U4'].mean() 
    df.loc[:,'iso1_asy_centered'] = df['iso1_asy'] - df['iso1_asy'].mean()   
    df.loc[:,'iso2_asy_centered'] = df['iso2_asy'] - df['iso2_asy'].mean()       
    df.loc[:,'iso3_asy_centered'] = df['iso3_asy'] - df['iso3_asy'].mean()      
    return df


In [37]:
#################################  prepare for _twoTime dataset  ##################################
#######  adding ori_subjName col, remove missing time points and centering  ########
# add ori_subjName column
combined = addOriSubj(combined)
combined_CAG_only_SCA = addOriSubj(combined_CAG_only_SCA)

# remove subjects with single time point
combined = removeSingleTimePointReIndex(combined)
combined_CAG_only_SCA = removeSingleTimePointReIndex(combined_CAG_only_SCA)

# centering values
combined_centered = centering_values(combined)
combined_CAG_only_SCA_centered =  centering_values(combined_CAG_only_SCA)

#################################  prepare for _oneTime dataset  ##################################
#######  adding ori_subjName col and centering  ########
# add ori_subjName column
combined_oneTime = addOriSubj(combined_oneTime)
combined_CAG_only_SCA_oneTime = addOriSubj(combined_CAG_only_SCA_oneTime)

# centering values
combined_centered_oneTime = centering_values(combined_oneTime)
combined_CAG_only_SCA_centered_oneTime =  centering_values(combined_CAG_only_SCA_oneTime)

In [51]:
###############################  compare LMM and GLM by computing LRT  ################################

# Model A: GLM on baseline only
#model_baseline = smf.ols('iso1 ~ CAG + Age_onset + C(side)', data=combined_CAG_only_SCA_centered_oneTime).fit()
model_baseline = smf.ols('iso1 ~ CAG + Age_onset + C(side)', data=combined_CAG_only_SCA_centered).fit()

# Model B: LMM on full data (two time points), using subject as a random effect
# must use maximum likelihood (ML) estimation (not REML) to compare models with different fixed effects or datasets.
model_full = smf.mixedlm('iso1 ~ CAG + Age_onset + C(side)', data=combined_CAG_only_SCA_centered, groups=combined_CAG_only_SCA_centered['ori_subjName']).fit(reml=False) 

# Compute Likelihood Ratio Statistic
lr_stat = 2 * (model_full.llf - model_baseline.llf)
df_diff = model_full.df_modelwc - model_baseline.df_model

p_value = stats.chi2.sf(lr_stat, df_diff)

print(f"LRT statistic = {lr_stat:.3f}, df = {df_diff}, p = {p_value:.4f}")



LRT statistic = 101.703, df = 2.0, p = 0.0000
